# Experiment: Draw Order Subset Signal Audit

Objective:
- Verify whether the official `drawNumberAppear` field is complete and safe for strictly prequential research.
- Test one preregistered Laplace position model against the exact uniform probability of the complete unordered six-number subset.

Success requires both games to have negative mean regret, a negative block-bootstrap upper bound, stable halves/recent windows, and final e-value at least 40. Historical results can never directly promote the model.


In [1]:
# Setup: imports and reproducibility
from __future__ import annotations

import json
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "research").is_dir():
    ROOT = next(parent for parent in ROOT.parents if (parent / "research").is_dir())
sys.path.insert(0, str(ROOT))

from research.draw_order_signal import run_draw_order_signal, validate_result

FORMAL_PATH = ROOT / "research" / "results" / "draw_order_signal.json"
ROOT


WindowsPath('%USERPROFILE%/Desktop/lotto-lab')

## Plan

- Hypothesis: past ball-by-position counts contain reproducible information about the next unordered six-number set.
- Model: one fixed cumulative position-specific Dirichlet model with `alpha=1`; no sweep.
- Leakage control: forecast draw *t* before updating with `drawNumberAppear_t`.
- Metrics: exact subset regret versus uniform, 13-draw block bootstrap, temporal stability, likelihood-ratio e-value, and a descriptive conditional-order permutation test.
- Decision: retain null-safe unless both games pass every frozen criterion.


In [2]:
# Inspect and validate the frozen formal artifact first.
formal = json.loads(FORMAL_PATH.read_text(encoding="utf-8"))
validate_result(formal)
quality_summary = {
    game: {
        "files": profile["files"],
        "draws": profile["draws"],
        "date_range": profile["date_range"],
        "draw_order_coverage_rate": profile["draw_order_coverage_rate"],
        "raw_ledger_mismatches": profile["raw_ledger_mismatches"],
    }
    for game, profile in formal["data_quality"]["games"].items()
}
quality_summary


{'super': {'files': 223,
  'draws': 1929,
  'date_range': ['2008-01-24', '2026-07-16'],
  'draw_order_coverage_rate': 1.0,
  'raw_ledger_mismatches': 0},
 'lotto649': {'files': 271,
  'draws': 2153,
  'date_range': ['2007-01-02', '2026-07-17'],
  'draw_order_coverage_rate': 1.0,
  'raw_ledger_mismatches': 0}}

## Results

The next cell recomputes all 4,082 prequential forecasts, 2,000 block-bootstrap samples per game, and 2,000 conditional-order permutations per game. It then requires exact equality with the frozen audit hash.


In [3]:
# Full reproducibility check against the formal artifact.
recomputed = run_draw_order_signal(base=ROOT)
assert recomputed["audit_hash"] == formal["audit_hash"]
result_rows = [
    {
        "game": game,
        "draws": row["draws"],
        "mean_regret_nats": row["mean_regret_nats"],
        "bootstrap_95_low": row["bootstrap_95_low"],
        "bootstrap_95_high": row["bootstrap_95_high"],
        "final_e_value": row["final_e_value"],
        "order_permutation_p": row["conditional_order_permutation"]["p_value"],
        "qualified": row["future_challenger_qualified"],
    }
    for game, row in recomputed["diagnostics"].items()
]
result_rows


[{'game': 'super',
  'draws': 1929,
  'mean_regret_nats': 0.024461981484985,
  'bootstrap_95_low': 0.0134437996348278,
  'bootstrap_95_high': 0.0354005183504131,
  'final_e_value': 3.21274164729037e-21,
  'order_permutation_p': 0.841079460269865,
  'qualified': False},
 {'game': 'lotto649',
  'draws': 2153,
  'mean_regret_nats': 0.030719204217656,
  'bootstrap_95_low': 0.0201867178973399,
  'bootstrap_95_high': 0.0417296554837092,
  'final_e_value': 1.88989451664267e-29,
  'order_permutation_p': 0.124437781109445,
  'qualified': False}]

## Next steps

- Decision: stop this hypothesis and retain the existing null-safe protocol.
- Do not add a draw-order Agent or watcher integration; both games have positive regret and no stable subset-probability advantage.
- Keep the field-quality checks as reusable guards if the official API schema changes.
- Only genuinely new, preregistered future information may open another challenger.
